In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt

from selenium.common.exceptions import TimeoutException
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException

from scipy.stats import gaussian_kde
import numpy as np
from datetime import datetime

from concurrent.futures import ThreadPoolExecutor

def extract_matches(soup):
    data = []
    for sublist in soup.find_all("div", class_="results-sublist"):
        date_element = sublist.find("span", class_="standard-headline")
        if date_element:
            date = date_element.text.strip().split()[-3:]  # the str is "Results for Month day year" therefore take only the last 3 elements of the list
        else:
            date = datetime.now().strftime("%B %d, %Y")  # Assign today's date when date is None
        for result_con in sublist.find_all("div", class_="result-con"):
            a_reset = result_con.find("a", class_="a-reset")
            if a_reset:
                result = a_reset.find("div", class_="result")

                # Extract team names, score, and match link
                team1 = result.find_all("td", class_="team-cell")[0].text.strip()
                score = result.find("td", class_="result-score").text.strip()
                team2 = result.find_all("td", class_="team-cell")[1].text.strip()
                match_link = "https://www.hltv.org" + a_reset["href"]
                event = result.find("td", class_="event")
                if event:
                    event_name = result.find("span", class_='event-name').text.strip()

                # Create a dictionary for each match with relevant data
                match_data = {
                    "Team 1": team1,
                    "Team 2": team2,
                    "Score": score,
                    "Match Link": match_link,
                    "Event Name": event_name,
                    "Date": date,
                }
                data.append(match_data)

    return data


def get_map_data(match_link):
    try:
        return extract_map_data(match_link)
    except Exception as e:
        print(f"Error while processing match link: {e}")
        return None

def extract_map_data(match_link):
    print(f"Processing match link: {match_link}")

    # Set up the WebDriver (without specifying the driver path)
    driver = webdriver.Firefox()

    # Navigate to the match page
    driver.get(match_link)

    try:
        # Wait for the flexbox-column element to appear (up to 10 seconds)
        wait = WebDriverWait(driver, 10)
        flexbox_column = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.flexbox-column")))

        print("Found flexbox-column")

        map_data = []

        # Step 1: Iterate through the child elements with <div class="mapholder">
        mapholder_index = 1
        for mapholder in flexbox_column.find_elements(By.CSS_SELECTOR, "div.mapholder"):
            print("Found mapholder")

            # Step 2: For each mapholder, find the child element with <div class="played">
            played = mapholder.find_element(By.CSS_SELECTOR, "div.played")

            if played:
                print("Found played")
                # Step 3: Inside the played element, find the child element with <div class="map-name-holder">
                map_name_holder = played.find_element(By.CSS_SELECTOR, "div.map-name-holder")

                if map_name_holder:
                    print("Found map-name-holder")
                    # Step 4: Extract the map name from the map-name-holder element
                    map_name_element = map_name_holder.find_element(By.CSS_SELECTOR, "div.mapname")

                    if map_name_element:
                        print("Found mapname")
                        map_name = map_name_element.text

                        # Wait for the score elements to appear (up to 10 seconds)
                        team1_score_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, f'div.mapholder:nth-child({mapholder_index}) div.results.played div[class^="results-left"] div.results-team-score')))
                        team2_score_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, f'div.mapholder:nth-child({mapholder_index}) div.results.played span[class^="results-right"] div.results-team-score')))

                        # Extract the scores (text from the elements)
                        team1_score = int(team1_score_element.text)
                        team2_score = int(team2_score_element.text)

                        # Create a dictionary with the map data
                        map_data.append({
                            "map_name": map_name,
                            "team1_score": team1_score,
                            "team2_score": team2_score
                        })

                        print(f"Appended map data: {map_data[-1]}")

                        mapholder_index += 1
    except NoSuchElementException:
        # Handle "Are you a robot?" page
        try:
            robot_button = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="CybotCookiebotDialogBodyButtonDecline"]')))
            print("Located 'I am not a robot' button")
            robot_button.click()
            print("Clicked 'I am not a robot' button")
        except Exception as e:
            print(f"Error while clicking 'I am not a robot' button: {e}")
    except Exception as e:
        print(f"Error while processing match link: {e}")
    finally:
        # Close the WebDriver
        driver.quit()

    return map_data

# Set up the WebDriver (without specifying the driver path)
driver = webdriver.Firefox()

# Navigate to the first page
url = "https://www.hltv.org/results"
driver.get(url)

# Wait for the cookie banner to appear (up to 10 seconds) and click it
try:
    wait = WebDriverWait(driver, 10)
    cookie_banner = wait.until(EC.presence_of_element_located((By.XPATH, "//*[@id='CybotCookiebotDialogBodyButtonDecline']")))
    cookie_banner.click()
except TimeoutException:
    print("Cookie banner not found. Continuing with scraping...")

# Scrape match history for a specified number of pages
num_pages = 12  # Change this value to the desired number of pages
match_history = []

for page in range(1, num_pages + 1):
    offset = (page - 1) * 100
    url = f"https://www.hltv.org/results?offset={offset}"

    driver.get(url)

    # Get the HTML content
    html_content = driver.page_source

    # Parse the HTML content with BeautifulSoup
    soup = BeautifulSoup(html_content, "html.parser")

    match_history.extend(extract_matches(soup)[:])

# Close the WebDriver
driver.quit()

# Create a pandas DataFrame with the scraped data
df = pd.DataFrame(match_history)

# Save the DataFrame to an Excel file
#df.to_excel("match_history.xlsx", index=False)

# Look at winrates by adding a column of who won
# Step 1: Create a new column 'Outcome'
def get_winner(score, team1, team2):
    s1, s2 = map(int, score.split(" - "))
    if s1 > s2:
        return team1
    if s1 < s2:
        return team2
    else:
        return "Draw"

df["Outcome"] = df[["Score", "Team 1", "Team 2"]].apply(lambda row: get_winner(row["Score"], row["Team 1"], row["Team 2"]), axis=1)

# Step 2: Create a new DataFrame 'bo1s' for rows with a score superior to 3 to account for matches that are either BO1 or BO5
def is_score_superior_to_3(score):
    s1, s2 = map(int, score.split(" - "))
    return s1 > 3 or s2 > 3

bo1s = df[df["Score"].apply(is_score_superior_to_3)].copy()
df = df[~df["Score"].apply(is_score_superior_to_3)]

# Use a ThreadPoolExecutor to parallelize the web scraping
with ThreadPoolExecutor(max_workers=8) as executor:
    # Map the get_map_data function to the match links
    map_data_results = list(executor.map(get_map_data, df["Match Link"]))

# Update the DataFrame with the results
df["Match Data"] = map_data_results



In [ ]:
# Flatten the list of dictionaries in the "Match Data" column
match_data_flattened = [item for sublist in df["Match Data"] for item in sublist]

# Create a new DataFrame from the flattened list of dictionaries
match_data_df = pd.DataFrame(match_data_flattened)

# Add the new DataFrame to your existing DataFrame
df = pd.concat([df, match_data_df], axis=1)

# Iterate through the rows of the dataframe
for i, row in df.iterrows():
    # Extract the dictionary from the "Match Data" column
    match_data = row['Match Data']
    # If the value is a float, skip to the next row
    if isinstance(match_data, float):
        continue
    # Iterate through the dictionaries in the list
    for j, data in enumerate(match_data, start=1):
        # Extract the map name and scores for both teams
        map_name = data['map_name']
        team1_score = data['team1_score']
        team2_score = data['team2_score']
        # Create new columns in the dataframe for each map played
        map_col_name = f"map{j}"
        team1_col_name = f"team1_score_map{j}"
        team2_col_name = f"team2_score_map{j}"
        # Fill in the map name and team scores for the appropriate row
        df.loc[i, map_col_name] = map_name
        df.loc[i, team1_col_name] = team1_score
        df.loc[i, team2_col_name] = team2_score

df.drop(columns=["map_name", "team1_score", "team2_score"], inplace=True)
df.dropna(axis=0, how='all', inplace=True)

In [ ]:
columns_to_drop = ['map4', 'team1_score_map4', 'team2_score_map4', 'map5', 'team1_score_map5', 'team2_score_map5']
df.drop(columns=columns_to_drop, inplace=True)
general_df = df.copy()
general_df

In [ ]:
general_df.to_excel("general_df_800.xlsx", index=False)